In [1]:
!pip install arxiv pymupdf pandas

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
   ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.1 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 4.1/4.1 MB 13.0 MB/s  0:00:00
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
   ---------------------------------------- 0.0/63.0 MB ? eta -:--:--
   -- ------------------------------------- 3.4/63.0 MB 28.5 MB/s eta 0:00:03
   --- ------------------------------------ 5.8/63.0 MB 16.9 MB/s eta 0:00:04
   ----- ---------------------------------- 8.9/63.0 MB 16.6 MB/s eta 0:00:04
   ------ --------------------------------- 10.5/63.0 MB 15.1 MB/s eta 0:00:04
   ------- -------------------------------- 12.1/63.0 MB 12.8 MB/s eta 0:00:04
   -------- ------------------------------- 13.1/63.0 MB 11.7 MB/s eta 0:00:05
   -------- ------------------------------- 13.9/63.0 M

In [2]:
import arxiv
import fitz  # pymupdf
import pandas as pd
import json
import os
import time

e:\Uni\information retrieval\.venv\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [3]:
CATEGORY = "cs.CL"
DATE_START = "202301010000"
DATE_END = "202412312359"
PAPERS_PER_TOPIC = 110   # a bit over 400/4 so failed downloads don't sink us

# sub-topic label -> keyword phrase searched in the abstract
SUBTOPICS = {
    "machine translation": "machine translation",
    "question answering": "question answering",
    "summarization": "summarization",
    "dialogue systems": "dialogue system",
    "sentiment analysis": "sentiment analysis",
}

PDF_DIR = "data/pdfs"
os.makedirs(PDF_DIR, exist_ok=True)

In [4]:
client = arxiv.Client(page_size=100, delay_seconds=3, num_retries=3)

def build_query(phrase):
    return (f'cat:{CATEGORY} AND abs:"{phrase}" '
            f'AND submittedDate:[{DATE_START} TO {DATE_END}]')

papers = []
seen_ids = set()

for label, phrase in SUBTOPICS.items():
    query = build_query(phrase)
    search = arxiv.Search(query=query, max_results=PAPERS_PER_TOPIC,
                          sort_by=arxiv.SortCriterion.Relevance)
    count = 0
    for result in client.results(search):
        paper_id = result.get_short_id()
        if paper_id in seen_ids:      # a paper can match two topics, keep the first
            continue
        seen_ids.add(paper_id)
        papers.append({
            "arxiv_id": paper_id,
            "title": result.title,
            "link": result.entry_id,
            "subtopic": label,
            "result": result,         # kept temporarily so we can download the pdf
        })
        count += 1
    print(label, "->", count, "papers")

machine translation -> 110 papers
question answering -> 110 papers
summarization -> 110 papers
dialogue systems -> 110 papers
sentiment analysis -> 110 papers


In [8]:
import requests

def download_pdf(url, path):
    # arXiv can reject requests with no user agent, so we send one
    headers = {"User-Agent": "course-project-student/1.0"}
    response = requests.get(url, headers=headers, timeout=60)
    response.raise_for_status()   # raises an error if the download failed
    with open(path, "wb") as f:
        f.write(response.content)

def get_pdf_text(paper):
    filename = paper["arxiv_id"].replace("/", "_") + ".pdf"
    path = os.path.join(PDF_DIR, filename)
    if not os.path.exists(path):
        download_pdf(paper["result"].pdf_url, path)
        time.sleep(1)  # be polite to arXiv
    doc = fitz.open(path)
    text = " ".join(page.get_text() for page in doc)
    return text, len(doc)

corpus = []
for i, paper in enumerate(papers):
    try:
        text, n_pages = get_pdf_text(paper)
    except Exception as e:
        print("failed:", paper["arxiv_id"], e)
        continue
    corpus.append({
        "arxiv_id": paper["arxiv_id"],
        "title": paper["title"],
        "link": paper["link"],
        "subtopic": paper["subtopic"],
        "n_pages": n_pages,
        "text": text,
    })
    if i % 25 == 0:
        print(i, "of", len(papers), "done")

0 of 550 done
25 of 550 done
MuPDF error: syntax error: could not parse color space (59 0 R)

50 of 550 done
75 of 550 done
100 of 550 done
125 of 550 done
150 of 550 done
175 of 550 done
MuPDF error: syntax error: could not parse color space (63 0 R)

MuPDF error: syntax error: could not parse color space (619 0 R)

200 of 550 done
225 of 550 done
250 of 550 done
275 of 550 done
300 of 550 done
325 of 550 done
350 of 550 done
375 of 550 done
400 of 550 done
425 of 550 done
450 of 550 done
475 of 550 done
500 of 550 done
525 of 550 done


In [9]:
def length_variant(n_pages):
    if n_pages <= 6:      # spec says <6 short and 7-9 medium, so 6 is undefined
        return "short"
    elif n_pages <= 9:
        return "medium"
    else:
        return "long"

for doc in corpus:
    doc["variant"] = length_variant(doc["n_pages"])

df = pd.DataFrame(corpus)
df.to_json("data/corpus.json", orient="records", indent=2)

print(len(df), "papers saved")
print(df["subtopic"].value_counts())
print(df["variant"].value_counts())

550 papers saved
subtopic
machine translation    110
question answering     110
summarization          110
dialogue systems       110
sentiment analysis     110
Name: count, dtype: int64
variant
long      390
medium     84
short      76
Name: count, dtype: int64


In [10]:
info = {
    "category": CATEGORY,
    "date_range": [DATE_START, DATE_END],
    "queries": {label: build_query(p) for label, p in SUBTOPICS.items()},
    "sort": "relevance",
    "n_papers": len(df),
}
with open("data/corpus_info.json", "w") as f:
    json.dump(info, f, indent=2)